In [3]:
!pip install langchain langchain-community chromadb sentence-transformers langsmith groq langchain-groq

     ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
     ---------------------------------------- 52.0/52.0 kB 2.6 MB/s eta 0:00:00
  Using cached grpcio-1.80.0-cp311-cp311-win_amd64.whl.metadata (3.9 kB)
     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 2.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB 4.8 MB/s eta 0:00:00
  Using cached googleapis_common_protos-1.74.0-py3-none-any.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/112.7 kB ? eta -:--:--
   ---------------------------------------- 112.7/112.7 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------- ----------------------------- 0.6/2.5 MB 20.1 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.5 MB 12.3 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import sys
from dotenv import load_dotenv
import mongoengine

load_dotenv()

# MongoDB connection
mongoengine.connect(
    db=os.getenv("MONGO_DB_NAME"),
    host=os.getenv("MONGO_URI")
)

sys.path.append(os.path.abspath(".."))

# Langsmith tracing setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "inventory-rag-week8"

print("✅ All imports and connections ready!")

✅ All imports and connections ready!


In [6]:
# NEW
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load all 3 text files
docs_path = "./"  # notebooks folder where files are saved

files = {
    "product_manual": "product_manual.txt",
    "return_policy": "return_policy.txt",
    "vendor_faq": "vendor_faq.txt"
}

raw_docs = []
for doc_name, filename in files.items():
    with open(f"{docs_path}{filename}", "r", encoding="utf-8") as f:
        content = f.read()
        raw_docs.append({
            "content": content,
            "source": doc_name
        })
    print(f"✅ Loaded {filename} ({len(content)} characters)")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # each chunk = 500 characters
    chunk_overlap=50,     # 50 character overlap between chunks
    separators=["\n\n", "\n", ".", " "]
)

all_chunks = []
for doc in raw_docs:
    chunks = text_splitter.split_text(doc["content"])
    for chunk in chunks:
        all_chunks.append({
            "text": chunk,
            "source": doc["source"]
        })

print(f"\n📄 Total chunks created: {len(all_chunks)}")
print(f"\n🔍 Sample chunk from product_manual:")
print(all_chunks[0]["text"])
print(f"\n🔍 Sample chunk from return_policy:")
sample = next(c for c in all_chunks if c["source"] == "return_policy")
print(sample["text"])

✅ Loaded product_manual.txt (15146 characters)
✅ Loaded return_policy.txt (5521 characters)
✅ Loaded vendor_faq.txt (7661 characters)

📄 Total chunks created: 76

🔍 Sample chunk from product_manual:
PRODUCT MANUAL & WARRANTY INFORMATION
Last Updated: April 2026
This document contains warranty information, product specifications,
and usage guidelines for all products in our inventory.

ELECTRONICS

🔍 Sample chunk from return_policy:
RETURN & REFUND POLICY
Last Updated: April 2026
This document outlines our complete return, refund, and exchange policy.

GENERAL RETURN POLICY


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
print("⏳ Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")

# Setup Chromadb
chroma_client = chromadb.Client()

# Delete collection if exists (clean start)
try:
    chroma_client.delete_collection("inventory_docs")
except:
    pass

collection = chroma_client.create_collection("inventory_docs")

# Embed and store all chunks
print(f"⏳ Embedding {len(all_chunks)} chunks...")

texts = [c["text"] for c in all_chunks]
sources = [c["source"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"source": s} for s in sources],
    ids=ids
)

print(f"\n✅ Successfully stored {collection.count()} chunks in Chromadb!")
print(f"   From product_manual : {sources.count('product_manual')} chunks")
print(f"   From return_policy  : {sources.count('return_policy')} chunks")
print(f"   From vendor_faq     : {sources.count('vendor_faq')} chunks")

⏳ Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded!
⏳ Embedding 76 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


✅ Successfully stored 76 chunks in Chromadb!
   From product_manual : 42 chunks
   From return_policy  : 15 chunks
   From vendor_faq     : 19 chunks


In [8]:
def retrieve_relevant_chunks(query, top_k=3):
    """
    Retrieve most relevant chunks from Chromadb for a given query
    """
    # Embed the query
    query_embedding = embedding_model.encode([query]).tolist()

    # Search Chromadb
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i]
        })

    return chunks

# Test the function
print("🔍 Testing retrieve_relevant_chunks()")
print("=" * 60)

test_queries = [
    "What's the return policy for damaged items?",
    "What is the warranty period for iPhone 14?",
    "What is the minimum order quantity?",
    "Can I return opened food products?"
]

for query in test_queries:
    print(f"\n❓ Query: '{query}'")
    chunks = retrieve_relevant_chunks(query, top_k=3)
    for i, chunk in enumerate(chunks, 1):
        print(f"  {i}. [{chunk['source']}] (distance: {chunk['distance']:.4f})")
        print(f"     {chunk['text'][:150]}...")
    print()

🔍 Testing retrieve_relevant_chunks()

❓ Query: 'What's the return policy for damaged items?'
  1. [return_policy] (distance: 0.6232)
DAMAGED ITEMS

1. Inspect all items immediately upon delivery.
2. Report...
  2. [return_policy] (distance: 0.7521)
     1. All products can be returned within 30 days of purchase date.
2. Items must be in original condition with original packaging.
3. Proof of purchase ...
  3. [return_policy] (distance: 0.7745)
NON-RETURNABLE ITEMS

The following items CANNOT be returned under any c...


❓ Query: 'What is the warranty period for iPhone 14?'
  1. [product_manual] (distance: 0.4801)
     iPhone 14 (Apple)
- Warranty: 1 year manufacturer warranty covering hardware defects and malfunctions.
- Screen damage, water damage, and physical dam...
  2. [product_manual] (distance: 0.6637)
     Apple iPhone 13 (Apple)
- Warranty: 1 year manufacturer warranty covering hardware defects.
- Screen and water damage not covered under standard warra...
  3. [product_manual

In [9]:
# Evaluation test cases for retrieval
RETRIEVAL_TEST_CASES = [
    {
        "query": "What's the return policy for damaged items?",
        "expected_source": "return_policy"
    },
    {
        "query": "What is the warranty for iPhone 14?",
        "expected_source": "product_manual"
    },
    {
        "query": "What is the minimum order quantity?",
        "expected_source": "vendor_faq"
    },
    {
        "query": "Can I return opened food products?",
        "expected_source": "return_policy"
    },
    {
        "query": "How long does delivery take?",
        "expected_source": "vendor_faq"
    },
    {
        "query": "What is the warranty on Samsung TV?",
        "expected_source": "product_manual"
    },
    {
        "query": "What is the restocking fee?",
        "expected_source": "vendor_faq"
    },
    {
        "query": "Can I exchange clothing items?",
        "expected_source": "return_policy"
    },
]

print("📊 Retrieval Eval Suite Results:")
print("=" * 60)

correct = 0
total = len(RETRIEVAL_TEST_CASES)

for case in RETRIEVAL_TEST_CASES:
    chunks = retrieve_relevant_chunks(case["query"], top_k=3)
    top_source = chunks[0]["source"]
    is_correct = top_source == case["expected_source"]
    if is_correct:
        correct += 1

    status = "✅" if is_correct else "❌"
    print(f"{status} Query: '{case['query']}'")
    print(f"   Expected: {case['expected_source']} | Got: {top_source}")
    print()

accuracy = correct / total * 100
print("=" * 60)
print(f"📈 Retrieval Accuracy: {correct}/{total} = {accuracy:.1f}%")

📊 Retrieval Eval Suite Results:
✅ Query: 'What's the return policy for damaged items?'
   Expected: return_policy | Got: return_policy

✅ Query: 'What is the warranty for iPhone 14?'
   Expected: product_manual | Got: product_manual

✅ Query: 'What is the minimum order quantity?'
   Expected: vendor_faq | Got: vendor_faq

✅ Query: 'Can I return opened food products?'
   Expected: return_policy | Got: return_policy

✅ Query: 'How long does delivery take?'
   Expected: vendor_faq | Got: vendor_faq

✅ Query: 'What is the warranty on Samsung TV?'
   Expected: product_manual | Got: product_manual

✅ Query: 'What is the restocking fee?'
   Expected: vendor_faq | Got: vendor_faq

✅ Query: 'Can I exchange clothing items?'
   Expected: return_policy | Got: return_policy

📈 Retrieval Accuracy: 8/8 = 100.0%
